# Aerial vehicle detector: training on Kaggle

Runs the same scripts as the Mac, on a free NVIDIA GPU.

**Before running:** Settings panel on the right → Accelerator: **GPU T4 x2**, Internet: **On**.

**Two modes**
1. Interactive, `SPEED_TEST = True`: click *Run All* to check everything works and see how long a full run would take (~15 min).
2. Background, `SPEED_TEST = False`: *Save Version* → *Save & Run All (Commit)*. It keeps running with the browser closed; results appear in the version's **Output** tab.

In [ ]:
SPEED_TEST = True   # set to False for the real training run
HOURS = 10          # training time budget; Kaggle stops sessions at ~12 h, so leave headroom
REPO = "https://github.com/krawat11/aerial-car-tracking.git"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Code and dataset go in `/tmp` (fast, not saved). Only `runs/` is linked into `/kaggle/working`,
so trained weights and logs are saved as the notebook's output even if the session stops early.

In [ ]:
import os
!rm -rf /tmp/proj && git clone -q $REPO /tmp/proj
os.chdir("/tmp/proj")
os.makedirs("/kaggle/working/runs", exist_ok=True)
!rm -rf runs && ln -s /kaggle/working/runs runs
!git log --oneline -3

Same Ultralytics version as the Mac, so results are comparable. Kaggle's own PyTorch is kept
because it is already matched to Kaggle's GPU drivers.

In [ ]:
!pip install -q "ultralytics==8.4.155" lap
import torch, ultralytics
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# download + convert VisDrone, then mask ignored regions, tag angles, oversample straight-down
!python scripts/prepare_visdrone.py --keep-zips
!python scripts/refine_visdrone.py

In [ ]:
if SPEED_TEST:
    !python scripts/train.py --name speedtest --epochs 1 --fraction 0.05
else:
    !python scripts/train.py --name visdrone_v1 --hours $HOURS

In [ ]:
if not SPEED_TEST:
    !python scripts/evaluate.py --model runs/train/visdrone_v1/weights/best.pt

Download from the **Output** tab: `runs/train/visdrone_v1/weights/best.pt` (the model),
plus `results.csv` / `results.png` (how accuracy changed while training).